In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [6]:
#hvem er nico

df = pd.read_csv("dropped_claims_train.csv")
X = df.iloc[:,1:]
y = df["ClaimNb"]
y

0         0
1         0
2         0
3         0
4         0
         ..
541411    0
541412    0
541413    0
541414    0
541415    0
Name: ClaimNb, Length: 541416, dtype: int64

In [47]:
#fan_int = features, fan_out = output
#Kaiming init/HE init
#Used to initialize weights for one hidden layer.
def he_init(fan_in, fan_out):
    std = np.sqrt(2 / fan_in)
    W = np.random.normal(0, std, size=(fan_out, fan_in))
    return W

#Activiation fucntion
def ReLu(arr):
    return np.maximum(0,arr)

#Cost function
def MSE(y_true,y_pred):
    return ((y_true-y_pred)**2).mean()

def forward_pass(X,W,b):
    #Combination
    z= X.dot(W.T)+b
    a = ReLu(z)
    return z,a


In [32]:
he_init(6,4)

array([[ 0.89321597, -0.57619931,  0.55207833, -0.51749603, -0.64396041,
        -0.11774584]])

***What this really needs is a way of remembering which weights and biases, and such corresponds to which layer***

**A ToDo, which I will do later/friday or sunday, is add this for the implementation and then try backpropagation**

$\frac{dL}{dZ2}=\frac{2}{m}*(A2-y_{true})$ <- **MSE Gradient**

In [9]:
""" 
Feed-forward
Get p number of features as input layer neurons. Decicde the number of hidden layers and neurons we have in each.
For our we have right now 1 hidden layers with p-2 neurons. For that we will get a linear combination
of weights and inputs for each neuron in the hidden layer. So each input neuron outputs to 4 neurons.

We use the ReLu function to return, a, which is the new values for input. We randomly init weights for that layer using
HE init. With all the a's we compute a linear combination for the output.


"""

#1 hidden layer
class neural:
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        output_size = 1

        self.W1 = np.random.normal(0, np.sqrt(2/input_size), (input_size, hidden_size))
        self.b1 = np.zeros((1, hidden_size))

        self.W2 = np.random.normal(0, np.sqrt(2/hidden_size), (hidden_size, output_size))
        self.b2 = np.zeros((1, output_size))

        self.cache = {}
    
 
    
    def MSE(self, y_true,y_pred):
        cost = ((y_true-y_pred)**2).mean()
        return cost
    
    def ReLu(self, Z):
        return np.maximum(0,Z)
    
    def grad_ReLu(self, Z):
        return (Z>0).astype(float)
    
    def forward_pass(self, X):

        #Layer 1
        Z1 = X.dot(self.W1) + self.b1
        A1 = self.ReLu(Z1)

        #Layer 2
        Z2 = A1.dot(self.W2) + self.b2
        A2 = Z2 #Linear activation

    
        #a would be the value for the hidden layer
        self.cache = {"X": X, "Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
        return A2

    def backward(self, y_true, learning_rate = 0.01):
        X = self.cache["X"]
        Z1 = self.cache["Z1"]
        A1 = self.cache["A1"]
        Z2 = self.cache["Z2"]
        A2 = self.cache["A2"]
        y_true = y_true.reshape(-1, 1)
        m = y_true.shape[0]


        #Output gradient/MSE, with respect to Z2
        #A2 would be the predicted value
        dZ2 = (2/m)*(A2-y_true)

        dW2 = A1.T.dot(dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m


        dA1 = dZ2.dot(self.W2.T)
        dZ1 = dA1 * self.grad_ReLu(Z1)

        dW1 = X.T.dot(dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m

        # update weights and biases
        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1
        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2
    def train(self, X, y, epochs = 1000, learning_rate = 0.01):
        X = np.array(X)
        y = np.array(y)
        m = X.shape[0]  # number of examples
        y = y.reshape(-1, 1)    

        for epoch in range(epochs):
            #Using the forward pass function
            A2 = self.forward_pass(X)

            #Monitoring the loss
            loss = np.mean((A2 - y)**2)

            #Doing the backward propagation
            self.backward(y, learning_rate)

            #Right now printing for every 100 epoch, the loss.
            if epoch % 100 == 0:
                print(f"Epoch {epoch}, Loss: {loss:.6f}")

    






In [7]:
if __name__ == "__main__":
    # Create dummy data
    X = np.random.randn(100000, 10)  # 100 samples, 10 features
    y = np.random.randn(100000)      # 100 targets (Rank 1 array)

    # Initialize
    nn = neural(input_size=10, hidden_size=5)
    
    # Train (Should run without crashing now)
    nn.train(X, y, epochs=500, learning_rate=0.01)

Epoch 0, Loss: 3.386515
Epoch 100, Loss: 3.386310
Epoch 200, Loss: 3.386104
Epoch 300, Loss: 3.385899
Epoch 400, Loss: 3.385694


In [13]:
new_n = neural(6,4)
new_n.train(X,y, epochs=2000)

Epoch 0, Loss: 221267.435246
Epoch 100, Loss: 32.660356
Epoch 200, Loss: 32.521623
Epoch 300, Loss: 32.393631
Epoch 400, Loss: 32.266451
Epoch 500, Loss: 32.140074
Epoch 600, Loss: 32.014495
Epoch 700, Loss: 31.889706
Epoch 800, Loss: 31.765700
Epoch 900, Loss: 31.642470
Epoch 1000, Loss: 31.520008
Epoch 1100, Loss: 31.398310
Epoch 1200, Loss: 31.277367
Epoch 1300, Loss: 31.157173
Epoch 1400, Loss: 31.037722
Epoch 1500, Loss: 30.919007
Epoch 1600, Loss: 30.801022
Epoch 1700, Loss: 30.683761
Epoch 1800, Loss: 30.567217
Epoch 1900, Loss: 30.451385


,0
0,-109.689961


X = range(10)